# 重写 Conv2dTransposeReshapeConcat 为 Conv2dReshapeConcatTranspose

In [1]:
from testing import viz_expr # 可视化 relay
from d2py.utils.file import mkdir
root_dir = ".temp"
mkdir(f"{root_dir}/logs")

In [2]:
import numpy as np
import onnx
import tvm
from tvm import relay

In [3]:
import torch
from torch.nn import functional as F
from torch import nn
from torch.onnx import OperatorExportTypes, utils

class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, 1, 1, 0, bias=False, groups=1)
        self.conv0 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)
        self.resize_2 = nn.Conv2d(64, 64, 1, 2, 0, bias=False, groups=1)
        self.conv1 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)
        self.resize_4 = nn.Conv2d(64, 64, 1, 4, 0, bias=False, groups=1)
        self.conv2 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)

    def forward(self, x):
        _x = self.conv(x)
        x0 = self.conv0(_x).transpose(1, 3).transpose(1, 2) # NCHW => NHWC
        x1 = self.conv1(self.resize_2(_x)).transpose(1, 3).transpose(1, 2)
        x2 = self.conv2(self.resize_4(_x)).transpose(1, 3).transpose(1, 2)
        x0 = x0.reshape(1, -1, 3)
        x1 = x1.reshape(1, -1, 3)
        x2 = x2.reshape(1, -1, 3)
        return torch.concat((x0, x1, x2), dim=1)

model = M()
model.eval()

shape = 1, 3, 48, 80
input_name = "data"
xx = torch.rand(*shape, dtype=torch.float32, requires_grad=False)
# model = torch.jit.trace(model, xx)
# 导出模型
output_name = "test"
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=17,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)

onnx_model = onnx.load(f"{root_dir}/{output_name}.onnx")
mod, params = relay.frontend.from_onnx(onnx_model, {input_name: shape}, freeze_params=True)
mod = relay.transform.InferType()(mod)
mod.show()

Exported graph: graph(%data : Float(1, 3, 48, 80, strides=[11520, 3840, 80, 1], requires_grad=0, device=cpu),
      %conv.weight : Float(64, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=1, device=cpu),
      %conv0.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %resize_2.weight : Float(64, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %conv1.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %resize_4.weight : Float(64, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %conv2.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu)):
  %/conv/Conv_output_0 : Float(1, 64, 48, 80, strides=[245760, 3840, 80, 1], requires_grad=0, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1], onnx_name="/conv/Conv"](%data, %conv.weight), scope: __main__.M::/torch.nn.modules.conv.Conv2d::conv # /media/pc/d